# 03 - HOG features + six traditional ML models
**Pipeline:** `64x64 RGB image -> grayscale -> HOG (1764 numbers) -> StandardScaler -> classifier`

All six classifiers receive **exactly the same features and the same train/test split**, so the comparison is fair.

![pipeline](../docs/00_project_pipeline.png)

| # | Model | Idea in one line |
|---|---|---|
| 1 | SVM | finds the maximum-margin boundary (RBF kernel) |
| 2 | KNN | votes among the k most similar training images |
| 3 | Random Forest | many decorrelated trees vote |
| 4 | Decision Tree | a single tree of if/else questions on features |
| 5 | Logistic Regression | linear scores + softmax probabilities |
| 6 | Naive Bayes | Bayes rule with independent Gaussian features |

In [ ]:
import sys; sys.path.append("..")          # lets the notebook import src/common.py
from src.common import *
import matplotlib.pyplot as plt, numpy as np, pandas as pd
print("Project root:", ROOT)
print("QUICK mode:", QUICK)
import time, joblib
from skimage.color import rgb2gray
from skimage.feature import hog
from skimage import exposure
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
from sklearn.metrics import precision_recall_fscore_support

X_train, X_test, y_train, y_test = load_split()
print("train", X_train.shape, "test", X_test.shape)

## Step 1 - What is HOG? (one image, step by step)
1. Convert to grayscale.
2. Compute the **gradient** (how fast intensity changes) in x and y -> magnitude + direction = edges.
3. Split the image into **8x8-pixel cells**; in each cell build a histogram of **9 gradient orientations**, weighted by magnitude.
4. Group cells into **2x2 blocks** and normalise (L2-Hys) -> robust to lighting.
5. Concatenate everything: 7x7 blocks x 4 cells x 9 bins = **1764 features** per image.

![HOG](../docs/00_hog_step_by_step.png)

In [ ]:
HOG_PARAMS = dict(orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), block_norm="L2-Hys")

img  = X_train[0]
gray = rgb2gray(img / 255.0)
gy, gx = np.gradient(gray)
magnitude   = np.hypot(gx, gy)
orientation = (np.rad2deg(np.arctan2(gy, gx)) % 180)
fd, hog_img = hog(gray, visualize=True, **HOG_PARAMS)

fig, ax = plt.subplots(1, 5, figsize=(17, 3.5))
ax[0].imshow(img);                                  ax[0].set_title(f"RGB 64x64 ({CLASSES[y_train[0]]})")
ax[1].imshow(gray, cmap="gray");                    ax[1].set_title("1. grayscale")
ax[2].imshow(magnitude, cmap="magma");              ax[2].set_title("2. gradient magnitude (edges)")
ax[3].imshow(orientation, cmap="hsv");              ax[3].set_title("2. gradient direction 0-180 deg")
ax[4].imshow(exposure.rescale_intensity(hog_img, in_range=(0, 10)), cmap="gray")
ax[4].set_title("3-4. HOG cells / blocks")
for a in ax: a.axis("off")
plt.tight_layout(); save_fig("03_hog_step_by_step.png"); plt.show()
print("HOG vector length:", fd.shape[0], "= 7 x 7 blocks x 2 x 2 cells x 9 orientations")

## Step 2 - Extract HOG for every image (same code for train and test)

In [ ]:
def extract_hog(images, batch=500):
    """images: (N,64,64,3) uint8 -> (N,1764) float32. Batched to save RAM."""
    out = []
    for s in range(0, len(images), batch):
        g = rgb2gray(images[s:s + batch] / 255.0)
        out.extend(hog(im, **HOG_PARAMS) for im in g)
    return np.asarray(out, dtype=np.float32)

t = time.time()
H_train = extract_hog(X_train); H_test = extract_hog(X_test)
print("H_train", H_train.shape, " H_test", H_test.shape, f"({time.time()-t:.0f}s)")
np.savez_compressed(PROC_DIR / "hog_features.npz", H_train=H_train, H_test=H_test,
                    y_train=y_train, y_test=y_test)

## Step 3 - Standardise
SVM, KNN and Logistic Regression are sensitive to feature scale. The scaler is **fitted on the training set only** (fitting on test data would leak information) and then applied to both.

In [ ]:
scaler = StandardScaler().fit(H_train)
S_train, S_test = scaler.transform(H_train), scaler.transform(H_test)
joblib.dump(scaler, MODELS_DIR / "hog_scaler.joblib")

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(H_train.ravel(), bins=60, color="gray");      ax[0].set_title("HOG values (raw)")
ax[1].hist(S_train.ravel(), bins=60, color="steelblue"); ax[1].set_title("after StandardScaler"); ax[1].set_xlim(-6, 6)
plt.tight_layout(); save_fig("03_scaling.png"); plt.show()

## Step 4 - Train and evaluate the six models
Helper that trains, times, evaluates (accuracy / precision / recall / F1 + confusion matrix) and saves each model.

In [ ]:
FAMILY = "Traditional ML"
results, preds = [], {}

def run(name, model):
    t = time.time(); model.fit(S_train, y_train); train_time = time.time() - t
    y_pred = model.predict(S_test); preds[name] = y_pred
    joblib.dump(model, MODELS_DIR / f"{slug(name)}.joblib")
    results.append(evaluate(name, FAMILY, y_test, y_pred, train_time))
    return model

def block_map(vec):
    """Sum a 1764-long HOG-sized vector over cells/orientations -> 7x7 spatial map."""
    return np.asarray(vec).reshape(7, 7, 2, 2, 9).sum(axis=(2, 3, 4))

## 4.1 - SVM
![SVM](../docs/model_steps/01_svm.png)

RBF-kernel SVM, `C=10`. Below: the 2-D PCA view of the HOG space and the number of support vectors per class (the images that define the boundary).

In [ ]:
svm = run("SVM", SVC(kernel="rbf", C=10, gamma="scale"))

pca = PCA(2, random_state=SEED).fit(S_train); P = pca.transform(S_train)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for i, cls in enumerate(CLASSES):
    ax[0].scatter(*P[y_train == i].T, s=5, alpha=.5, label=cls, color=plt.cm.tab10.colors[i])
ax[0].scatter(*P[svm.support_].T, s=18, facecolors="none", edgecolors="k", linewidths=.3)
ax[0].set_title("HOG features in 2-D PCA (circled = support vectors)"); ax[0].legend(markerscale=3, fontsize=7)
ax[1].bar(CLASSES, svm.n_support_, color=plt.cm.tab10.colors[:8]); ax[1].set_title("support vectors per class")
plt.setp(ax[1].get_xticklabels(), rotation=30); plt.tight_layout(); save_fig("03_svm_process.png"); plt.show()

## 4.2 - KNN
![KNN](../docs/model_steps/02_knn.png)

The number of neighbours `k` is chosen by 3-fold cross-validation **on the training set only**.

In [ ]:
ks = [1, 3, 5, 7, 9, 11, 15]
cv = [cross_val_score(KNeighborsClassifier(k), S_train, y_train, cv=3).mean() for k in ks]
best_k = ks[int(np.argmax(cv))]
plt.figure(figsize=(6, 3.5)); plt.plot(ks, cv, "o-"); plt.axvline(best_k, color="r", ls="--")
plt.xlabel("k"); plt.ylabel("CV accuracy"); plt.title(f"KNN: choosing k (best k = {best_k})")
plt.tight_layout(); save_fig("03_knn_k_selection.png"); plt.show()
knn = run("KNN", KNeighborsClassifier(n_neighbors=best_k))

## 4.3 - Random Forest
![Random Forest](../docs/model_steps/03_random_forest.png)

200 trees, each trained on a bootstrap sample with random feature subsets. The importance map shows *which image regions* the forest relies on.

In [ ]:
rf = run("Random Forest", RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED))

imp = block_map(rf.feature_importances_)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(X_train[0]); ax[0].set_title("example image"); ax[0].axis("off")
im = ax[1].imshow(imp, cmap="hot"); ax[1].set_title("RF importance per HOG block (7x7)")
plt.colorbar(im, ax=ax[1], fraction=.046); plt.tight_layout(); save_fig("03_rf_importance.png"); plt.show()

## 4.4 - Decision Tree
![Decision Tree](../docs/model_steps/04_decision_tree.png)

A single tree. Depth is selected by 3-fold CV on the training set (an unlimited tree memorises the training data).

In [ ]:
depths = [3, 5, 8, 12, 16, None]
cv = [cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=SEED), S_train, y_train, cv=3).mean() for d in depths]
best_d = depths[int(np.argmax(cv))]
plt.figure(figsize=(6, 3.5)); plt.plot([str(d) for d in depths], cv, "o-")
plt.xlabel("max_depth"); plt.ylabel("CV accuracy"); plt.title(f"Decision tree: choosing depth (best = {best_d})")
plt.tight_layout(); save_fig("03_dt_depth_selection.png"); plt.show()

dt = run("Decision Tree", DecisionTreeClassifier(max_depth=best_d, random_state=SEED))
plt.figure(figsize=(16, 6)); plot_tree(dt, max_depth=2, class_names=CLASSES, filled=True, fontsize=7, impurity=False)
plt.title("Top 3 levels of the tree (X[i] = i-th HOG feature)"); save_fig("03_dt_tree_top.png"); plt.show()

## 4.5 - Logistic Regression
![Logistic Regression](../docs/model_steps/05_logistic_regression.png)

Multinomial logistic regression: one weight vector per class, softmax turns the 8 scores into probabilities. The maps show where in the image each class's weights are large.

In [ ]:
lr = run("Logistic Regression", LogisticRegression(max_iter=1000, C=1.0))

maps = np.abs(lr.coef_).reshape(len(CLASSES), 7, 7, 2, 2, 9).mean(axis=(3, 4, 5))
fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
for a, m, cls in zip(axes.ravel(), maps, CLASSES):
    a.imshow(m, cmap="viridis"); a.set_title(cls); a.axis("off")
plt.suptitle("|weights| per HOG block for each class"); plt.tight_layout()
save_fig("03_lr_weights.png"); plt.show()

## 4.6 - Naive Bayes
![Naive Bayes](../docs/model_steps/06_naive_bayes.png)

Gaussian Naive Bayes: for each class it learns a mean and variance per HOG feature, then multiplies the (assumed independent) likelihoods with the class prior.

In [ ]:
nb = run("Naive Bayes", GaussianNB())

maps = nb.theta_.reshape(len(CLASSES), 7, 7, 2, 2, 9).mean(axis=(3, 4, 5))
fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
for a, m, cls in zip(axes.ravel(), maps, CLASSES):
    a.imshow(m, cmap="coolwarm", vmin=-.5, vmax=.5); a.set_title(cls); a.axis("off")
plt.suptitle("Mean (standardised) HOG energy per block learned for each class"); plt.tight_layout()
save_fig("03_nb_class_means.png"); plt.show()

## Step 5 - Compare the six traditional models

In [ ]:
df = pd.DataFrame(results).sort_values("f1_score", ascending=False).reset_index(drop=True)
display(df.round(4))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(df)); w = 0.2
for i, m in enumerate(["accuracy", "precision", "recall", "f1_score"]):
    ax.bar(x + (i - 1.5) * w, df[m], w, label=m)
ax.set_xticks(x); ax.set_xticklabels(df["model"], rotation=25); ax.set_ylim(0, 1); ax.legend(ncol=4, fontsize=8)
ax.set_title("Traditional ML (HOG) - test metrics"); plt.tight_layout(); save_fig("03_traditional_comparison.png"); plt.show()

# per-class F1
f1 = pd.DataFrame({n: precision_recall_fscore_support(y_test, p, labels=range(len(CLASSES)), zero_division=0)[2]
                   for n, p in preds.items()}, index=CLASSES)
plt.figure(figsize=(8, 4.5)); plt.imshow(f1.values, cmap="YlGn", vmin=0, vmax=1)
plt.xticks(range(f1.shape[1]), f1.columns, rotation=30, ha="right"); plt.yticks(range(len(CLASSES)), CLASSES)
for i in range(f1.shape[0]):
    for j in range(f1.shape[1]): plt.text(j, i, f"{f1.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("F1-score per class"); plt.colorbar(); plt.tight_layout(); save_fig("03_f1_per_class.png"); plt.show()

## Next
`04_deep_learning.ipynb` trains the MLP and the CNN directly on the images. All metrics are already stored in `results/metrics/`.